# Forest and boosting: what drives the errors?

Existing nested **held-network** predictions, trained on full centroid shift, with the one-minute tuning floor. No new models fitted. Each network has equal weight. Peak results reuse centroid predictions; they are not separately trained peak regressions.

**Findings:** forest and boosting residual ranks correlate at 0.793; their error signs agree in 86.2% of comparisons (these two descriptive numbers are pair-weighted). The problem is largely shared. Full compounds contribute 74.4% of weighted squared error. Network 37 / CB001_bubble_3 alone contributes 40.2%. Conversely, shifts of 1–5 minutes contribute 49.37 percentage points of the total 68.32% MAPE (72.3%). Near-unchanged cases contribute 28.61 points of the 71.61% floored relative score, but are excluded from ordinary MAPE.

The length ratio already separates delays from unchanged cases to some extent: descriptive AUC 0.778 pooled with equal-example weights, 0.850 averaged within the eight networks containing both classes. This is **not** a validated classification result. Mean-width ratio distinguishes changed from unchanged cases better than it distinguishes delay from unchanged. No new variable is justified by these associations alone.

**Next:** test a buffered two-stage model against the existing forest, retaining whole-network outer validation and choosing classifier thresholds within inner folds. Keep a separate review of extreme full-compound errors. These are exploratory results, not an untouched final test.

In [2]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import plotly.express as px
ROOT=Path.cwd()
while not (ROOT/'RAPID').exists():
    if ROOT==ROOT.parent: raise RuntimeError('Run inside the river-hierarchy repository')
    ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'RAPID/src'))
OUT=ROOT/'RAPID/outputs/hydrograph_relative_error_review_20261019'
rows=pd.read_parquet(OUT/'error_rows.parquet')
a=pd.read_csv(OUT/'attribution.csv',dtype={'category':str})
paired=pd.read_csv(OUT/'paired_errors.csv',dtype={'example':str})
selection=(a.scenario=='all')&(a.model=='forest')&(a.response=='centroid_shift_hours')&(a.buffer_minutes==1)
a=a[selection]
display(pd.read_csv(OUT/'metrics.csv').query("scenario=='all' and buffer_minutes==1"))

,scenario,model,response,buffer_minutes,n,r2,relative_error_pct,mape_pct,mae
0,all,boosting,centroid_shift_hours,1.0,2170,0.566516,80.324957,78.472863,3.037261
3,all,boosting,peak_shift_hours,1.0,2170,0.482464,87.962871,68.012438,4.634069
6,all,forest,centroid_shift_hours,1.0,2170,0.580049,71.609316,68.323060,2.937865
9,all,forest,peak_shift_hours,1.0,2170,0.495511,79.394936,63.001386,4.502415


## Which errors drive MAPE and R²?

MAPE excludes |shift| ≤ buffer and reweights networks within that subset. `mape_points` add to the overall MAPE; they are not subgroup MAPEs. `relative_points` add to the floored score including unchanged cases.

R² = 1 − SSE/SST. `sse_share_pct` is the share of weighted squared error; `sst_share_pct` is the share of weighted response variation around the **global weighted mean**. `r2_loss` sums to 1−R². Large shifts can dominate both numerator and denominator. These contributions do not tell us what R² would become after removing a case.

In [4]:
for grouping in ['shift_size','group','edit_type','example']:
    print(grouping)
    display(a[a.by==grouping][['category','n','mape_points','relative_points','sse_share_pct','sst_share_pct','r2_loss']].sort_values('mape_points',ascending=False))
fig=px.scatter(paired,x='y',y='prediction',color='group',symbol='edit_type',hover_data=['example','state','aliases','error','error_boosting','mape_points','sse_share_pct'],labels={'y':'Modelled centroid shift (min)','prediction':'Forest prediction (min)'})
lo=min(paired.y.min(),paired.prediction.min());hi=max(paired.y.max(),paired.prediction.max())
fig.add_shape(type='line',x0=lo,y0=lo,x1=hi,y1=hi,line=dict(dash='dash',color='gray'))
fig.show()

shift_size
group
edit_type
example


,category,n,mape_points,relative_points,sse_share_pct,sst_share_pct,r2_loss
185,"(1.0, 5.0]",731,49.374190,28.745539,4.122247,4.381869,0.017311
187,"(5.0, 15.0]",398,13.816253,10.380421,9.043175,3.310455,0.037977
186,"(15.0, 60.0]",156,4.929048,3.720120,34.798770,42.472024,0.146138
188,"(60.0, inf]",12,0.203569,0.153618,49.577619,44.303094,0.208202
184,"(-0.001, 1.0]",873,0.000000,28.609618,2.458189,5.532557,0.010323


,category,n,mape_points,relative_points,sse_share_pct,sst_share_pct,r2_loss
181,advance,1165,59.966747,38.759526,89.691227,86.369270,0.376659
182,delay,132,8.356313,4.240172,7.850584,8.098172,0.032969
183,unchanged,873,0.000000,28.609618,2.458189,5.532557,0.010323


,category,n,mape_points,relative_points,sse_share_pct,sst_share_pct,r2_loss
177,Compound member,1467,33.409533,43.231233,14.621536,17.786137,0.061403
180,Simple,258,14.464544,13.779458,5.091529,3.136901,0.021382
178,Full compound,177,13.434049,7.816165,74.380965,70.966809,0.312363
179,Island,268,7.014934,6.782460,5.905970,8.110153,0.024802


,category,n,mape_points,relative_points,sse_share_pct,sst_share_pct,r2_loss
173,48,72,11.571052,11.398038,1.873981,1.104311,0.007870
168,22,408,7.703769,6.418375,3.626580,4.152638,0.015230
174,54,81,7.009057,9.754982,4.574725,1.164965,0.019212
169,37,395,5.877061,4.270507,42.775839,28.820304,0.179637
167,21,30,5.771022,5.771022,2.053114,0.466792,0.008622
166,15,262,4.575234,4.246467,2.600881,6.524610,0.010922
172,46,223,4.194878,5.122554,9.645241,11.488200,0.040505
170,41,181,4.153122,4.820194,9.022318,10.071367,0.037889
162,02,35,3.877222,5.029184,7.101605,2.152037,0.029823
175,55,212,3.105496,3.151665,5.407895,7.775389,0.022710


## Existing geometry and response groups

D is the outer comparison domain; ratios are child/parent. `R_muL`, `R_sigmaL`, `R_W`, `R_CVW`, `R_H` represent mean travelled length, length SD, mean width, width CV and route entropy. `d_Dp` is parent dominance and `r_in` exposure. The model transforms ratios with ln(1+R); domain size is μL,Dp / 1 km. The table below evaluates individual geometric values descriptively, without fitting a classifier. AUC below 0.5 indicates the reverse direction. Network composition can influence pooled values, so within-network averages are included. None of the response-derived groups or diagnostic flags is an input predictor.

In [6]:
display(pd.read_csv(OUT/'geometry_separation.csv'))
display(pd.read_csv(OUT/'delay_vs_unchanged_geometry.csv'))
display(pd.read_csv(OUT/'response_group_counts.csv'))
for feature,label in [('log1p__B_ratio_route_length_mean','R_muL'),('log1p__B_ratio_width_mean','R_W'),('log1p__B_ratio_route_entropy','R_H')]:
    z=paired.copy();z[label]=np.expm1(z[feature])
    px.box(z,x='group',y=label,color='group',points=False,title='Pair distributions; not equal-network weighted').show()

,feature,symbol,category,pooled_auc,within_example_auc_mean,examples
0,log1p__B_ratio_route_length_mean,ln(1+R_muL),unchanged,0.791197,0.792989,13
1,log1p__B_ratio_route_length_mean,ln(1+R_muL),delay,0.910693,0.904222,8
2,domain_size,muL_Dp / 1 km,unchanged,0.706558,0.634139,13
3,domain_size,muL_Dp / 1 km,delay,0.742233,0.668244,8
4,log1p__B_ratio_route_length_sd,ln(1+R_sigmaL),unchanged,0.692348,0.677007,13
5,log1p__B_ratio_route_length_sd,ln(1+R_sigmaL),delay,0.769494,0.745596,8
6,log1p__B_ratio_width_mean,ln(1+R_W),unchanged,0.142646,0.182971,13
7,log1p__B_ratio_width_mean,ln(1+R_W),delay,0.213305,0.323886,8
8,log1p__B_ratio_width_cv,ln(1+R_CVW),unchanged,0.671845,0.616306,13
9,log1p__B_ratio_width_cv,ln(1+R_CVW),delay,0.631205,0.617650,8


,feature,symbol,pooled_auc,within_example_auc_mean,examples
0,log1p__B_ratio_route_length_mean,ln(1+R_muL),0.778215,0.849793,8
1,domain_size,muL_Dp / 1 km,0.601917,0.539901,8
2,log1p__B_ratio_route_length_sd,ln(1+R_sigmaL),0.678232,0.695165,8
3,log1p__B_ratio_width_mean,ln(1+R_W),0.459389,0.545281,8
4,log1p__B_ratio_width_cv,ln(1+R_CVW),0.570808,0.588910,8
5,log1p__B_ratio_route_entropy,ln(1+R_H),0.775653,0.666568,8
6,B_parent_dominance_mean,d_Dp,0.737057,0.547219,8
7,B_exposure,r_in,0.525845,0.455792,8


,group,advance,delay,unchanged
0,advance,1022,7,136
1,delay,10,80,42
2,unchanged,234,39,600


## A short list of hydrographs to review

- **37 / CB001_bubble_3:** dominates squared error; both models underpredict its very large advance.
- **48 / CB008_bubble_13:** a small advance with a large relative error, shared by both models.
- **02 / S013_unit_20:** centroid delay predicted as an advance by both models. Compare centroid and peak timing explicitly.
- **54 / S036_unit_52:** almost unchanged full centroid, but both models predict a delay. Example 54 has a broad, multi-crest event and a higher ending equilibrium; full-centroid metrics include that finite-window transition.

Plots use the hash-verified outlet series recorded in the finalized response tables on the same UTC clock, with no peak alignment. The numerical response table retains the finalized 0.5% window definitions. Plots are restricted to that shared window. Shape differences and crest dominance remain supporting diagnostics, not replacement response definitions.

In [8]:
from rapid_tools.hydrograph_response import load, pulse, distribution
import plotly.graph_objects as go
states,responses=load('0.5%')
cases=[('37','CB001_bubble_3'),('48','CB008_bubble_13'),('02','S013_unit_20'),('54','S036_unit_52')]
for ex,state in cases:
    r=paired[(paired.example==ex)&(paired.state==state)].iloc[0]
    display(r[['example','state','aliases','y','prediction','error_boosting','sse_share_pct','mape_points','peak_shift_hours','duration80_change_pct','asymmetry_change','peak_excess_change_pct']].to_frame('value'))
    source=responses[(responses.example==ex)&(responses.state==state)].iloc[0]
    moments=[];peaks=[];fig=go.Figure()
    for label,key,color in [('Parent','S000_base','#2563eb'),('Child',state,'#dc2626')]:
        sr=states[(states.example==ex)&(states.state==key)].iloc[0]
        t,q,_=pulse(sr)
        moments.append(distribution(t,q)['centroid_seconds']);peaks.append(t[q.argmax()])
        fig.add_scatter(x=pd.to_datetime(t,unit='s',utc=True),y=q+sr.baseline_cms,name=label,line=dict(color=color))
        fig.add_vline(x=moments[-1]*1000,line_color=color,line_dash='dot')
    np.testing.assert_allclose((moments[1]-moments[0])/3600,r.centroid_shift_hours,atol=1e-8)
    np.testing.assert_allclose((peaks[1]-peaks[0])/3600,r.peak_shift_hours,atol=1e-8)
    fig.update_layout(title=f'{ex} / {state} — finalized shared window; dotted lines: full centroids',xaxis_title='Time (UTC)',yaxis_title='Discharge (m³/s)',template='plotly_white')
    fig.show()


,value
example,37
state,CB001_bubble_3
aliases,CB001_bubble_3
y,-447.688034
prediction,-85.02111
error_boosting,374.340089
sse_share_pct,40.178198
mape_points,0.035298
peak_shift_hours,-8.75
duration80_change_pct,8.834285


,value
example,48
state,CB008_bubble_13
aliases,CB008_bubble_13
y,-1.487775
prediction,-6.215753
error_boosting,-3.812731
sse_share_pct,0.037462
mape_points,0.921126
peak_shift_hours,-0.033333
duration80_change_pct,0.040378


,value
example,02
state,S013_unit_20
aliases,S013_unit_20 | island_02_021
y,2.690794
prediction,-5.6322
error_boosting,-7.819258
sse_share_pct,0.238816
mape_points,0.824836
peak_shift_hours,0.066667
duration80_change_pct,-0.044009


,value
example,54
state,S036_unit_52
aliases,S036_unit_52 | island_54_053
y,0.041148
prediction,31.017803
error_boosting,10.075382
sse_share_pct,1.429406
mape_points,0.0
peak_shift_hours,-0.016667
duration80_change_pct,0.000532
